# 101_04 — Evaluación predictiva · Escenario A-1 · **barrido en $M$**

Paso **3b** del ciclo. Supone que `101_03_convergencia` dio veredicto OK **en
cada punto del barrido**; si las cadenas de un $M$ no convergieron, sus cifras
aquí no significan nada.

## Qué se reporta

- Predicción funcional a $h=1$ y su intervalo de credibilidad — se PERSISTEN, es lo que consume `101_05`
- **Bloque A** (MAE, RMSE, error máximo) sobre la ventana móvil
- **Bloque B** (Winkler, cobertura simultánea, cobertura puntual, MPIW) sobre la ventana móvil
- Muestra de predicciones: extractos regulares del test y las peores ventanas, con la curva anterior al lado
- Probabilidades posteriores de inclusión (PIP) contra la estructura del generador
- Cuadro resumen: mínimo, máximo y promedio de cada métrica sobre la ventana
- Monitoreo univariado por componente FPCA
- **§10: evaluación puntual de los scores (coeficientes)** — banda de credibilidad de cada $\xi_k$, métricas de intervalo sobre ella, y dispersión $\hat\xi$ contra $\xi$

Todo el cálculo vive en `fit/` y todo el dibujo en `graphics/`: este notebook
sólo orquesta.


## 1. Imports, rutas y artefactos

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from model_psbp_fd.pipelines import (
    cargar_curvas, cargar_curvas_true, cargar_representacion,
    cargar_fpca, cargar_estandarizador,
    cargar_datasets_ar, cargar_hiperparametros, cargar_config_evaluacion,
)
from model_psbp_fd.functions_models import DataStandardizer
from model_psbp_fd.models.pspb_fd_v3 import (
    PropagadorFuncional,
    ruta_traza, leer_traza, ModeloTraza,   # convencion de trazas (§2)
)

from model_psbp_fd.fit import (
    mise,                                    # piso de truncamiento (§3)
    intervalo_muestral,                      # banda sobre los scores (§3)
    agrupar_momentos,                        # media/sd entre cadenas (§3)
    ventana_movil_scores, ventana_movil_funcional,   # el motor de §4/5/9
    indicador_cobertura,                     # I_t puntual (§8)
    indicador_cobertura_simultanea,          # I_t simultaneo (§8)
    matriz_pip, contraste_con_verdad,        # §7
)
from model_psbp_fd.graphics import (
    plot_ventana_movil, plot_extractos_curvas,
)
from model_psbp_fd.utils import get_project_root

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline


In [ ]:
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# [CONFIG] 
BASENAME, ESCENARIO_ID, REPLICA_ID = "escenario_101", 1, 1
M_FPCA_LIST = (1, 2, 3, 4)

SALTAR_M_SIN_TRAZAS = True

PESOS_TAU = None

METRICA_PEORES = "rmse_f"   # cualquiera de METRICAS_MONITOR (nivel funcional)
N_PEORES = 5

M_FPCA_LIST = tuple(int(m) for m in M_FPCA_LIST)
assert len(set(M_FPCA_LIST)) == len(M_FPCA_LIST), "M_FPCA_LIST tiene repetidos."

EXPERIMENT_BASE = f"{BASENAME}_{ESCENARIO_ID}_r{REPLICA_ID:02d}"


def experiment_id(M: int) -> str:
    return f"{EXPERIMENT_BASE}_m{M:02d}"


def rutas(M: int) -> dict:
    eid = experiment_id(M)
    p = {
        "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw" / eid,
        "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / eid,
        "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict" / eid,
        "out_report":   PROJECT_ROOT / "reports" / "simulaciones" / eid,
        "out_artefact": PROJECT_ROOT / "artefact" / "simulaciones" / eid,
    }
    for r in p.values():
        r.mkdir(parents=True, exist_ok=True)
    return p


PATHS_M      = {M: rutas(M) for M in M_FPCA_LIST}
PATH_BARRIDO = PROJECT_ROOT / "reports" / "simulaciones" / f"{EXPERIMENT_BASE}_barrido_M"
PATH_BARRIDO.mkdir(parents=True, exist_ok=True)

print(f"barrido en M : {list(M_FPCA_LIST)}")
for M in M_FPCA_LIST:
    print(f"  M={M}  ->  {experiment_id(M)}")
print(f"salidas cruzadas de M : {PATH_BARRIDO}   "
      "(ventana movil por M, prefijo 88_*)")


# Las 10 metricas del marco teorico 02_02_03, en su orden. No se agregan otras.
METRICAS_A = [
    (u"1. MAE  (L^1)",                     "mae_f"),
    (u"2. RMSE (L^2)",                     "rmse_f"),
    (u"3. E_max promedio por curva",       "linf_medio"),
    (u"4. E_max peor de la ventana",       "linf_max"),
]
METRICAS_B = [
    (u"5. MPIW",                           "mpiw"),
    (u"6. PICP puntual",                   "picp"),
    (u"7. PICPB curva completa",           "picp_simultaneo"),
    (u"8. Winkler",                        "winkler"),
    (u"9. Winkler max promedio",           "winkler_max_medio"),
    (u"10. Winkler max peor ventana",      "winkler_max_glob"),
]
METRICAS_MONITOR = METRICAS_A + METRICAS_B

print("")
print("metricas monitoreadas (nivel funcional, en orden):")
for _et, _cw in METRICAS_MONITOR:
    print("   %-40s  columna=%s" % (_et, _cw))
print("   + I_t puntual y simultaneo (series por origen)  ->  "
      "67_indicador_cobertura.csv")

# Columnas de cada bloque: §4 dibuja el A y §5 el B, sin mezclarlos.
METRICAS_A_COLS = [c for _, c in METRICAS_A]
METRICAS_B_COLS = [c for _, c in METRICAS_B]



In [ ]:
# Estado por punto del barrido. Nada de lo que sigue usa variables globales del
# experimento: todo se lee de EST[M].
EST = {}
saltados = []

for M in M_FPCA_LIST:
    P = PATHS_M[M]
    if not (P["functional"] / "datasets_manifest.json").exists():
        saltados.append((M, "sin datasets_manifest.json - falta correr 101_01"))
        continue

    dfs_train, manifest = cargar_datasets_ar(P, bloque="train")
    dfs_test,  _        = cargar_datasets_ar(P, bloque="test")
    hp_json     = cargar_hiperparametros(P)
    P["trazas"] = P["out_artefact"].parent / hp_json.get("trazas_en", P["out_artefact"].name)
    eval_config = cargar_config_evaluacion(P)

    component_idx = manifest["component_idx"]
    n_components  = len(component_idx)
    assert n_components == M, (
        f"{experiment_id(M)} declara M={M} pero su manifest tiene "
        f"{n_components} componentes.")
    assert manifest["scores_scale"] == "raw_fpca_scores"

    EST[M] = {
        "paths": P, "eid": experiment_id(M),
        "dfs_train": dfs_train, "dfs_test": dfs_test, "manifest": manifest,
        "hp_json": hp_json, "eval_config": eval_config,
        "component_idx": component_idx, "n_components": n_components,
        "cov_names": manifest["cov_names"],
        "cov_por_componente": [list(manifest["cov_por_componente"][str(k)]) for k in range(n_components)],
        "n_lags": int(manifest["n_lags"]),
        "T": int(manifest["T"]), "T0": int(manifest["T0"]),
        "n_iter": int(hp_json["n_iter"]),
        "mcmc_cfg": hp_json["mcmc_config"],
        "burn": int(hp_json["mcmc_config"]["burn"]),
        # Parámetros de evaluación: se LEEN del artefacto, no se redeclaran.
        "nivel": float(eval_config.get("nivel_credibilidad", 0.95)),
        "modo_residuo": eval_config.get("modo_residuo", "ninguno"),
        "objetivo": eval_config.get("objetivo_evaluacion", "curva_suavizada"),
        "objetivo_sec": eval_config.get("objetivo_secundario", "representacion_fpca"),
        "ventanas_w": list(eval_config.get("ventana_movil", {}).get("w", [20, 30, 40])),
    }
    EST[M]["n_post"] = EST[M]["mcmc_cfg"]["nsim"] - EST[M]["burn"]

assert EST, "Ningún punto del barrido tiene datos. Ejecuta 101_01 primero."

for M, e in EST.items():
    print(f"M={M}  T={e['T']} T0={e['T0']} n_lags={e['n_lags']} "
          f"componentes={e['n_components']}  nivel={e['nivel']}  "
          f"modo_residuo={e['modo_residuo']!r}  objetivo={e['objetivo']!r}")
for M, motivo in saltados:
    print(f"  ! M={M} saltado: {motivo}")

# La comparación entre M sólo tiene sentido si el diseño de observación, la
# partición y el objetivo de evaluación son los mismos en todos los puntos: si
# no, las diferencias mezclarían el efecto de M con el de otra cosa.
_comun = {M: (e["T"], e["T0"], e["n_lags"], e["nivel"], e["modo_residuo"],
              e["objetivo"], tuple(e["ventanas_w"]), e["n_iter"],
              e["mcmc_cfg"]["nsim"], e["mcmc_cfg"]["N"], e["burn"])
          for M, e in EST.items()}
if len(set(_comun.values())) > 1:
    print("\n! Los puntos del barrido NO comparten diseño/evaluación:")
    for M, v in _comun.items():
        print(f"    M={M}: {v}")
    raise AssertionError(
        "El barrido en M exige T, T0, n_lags, nivel, modo_residuo, objetivo, "
        "ventanas y configuración MCMC idénticos. Regenera los puntos que "
        "difieren antes de comparar.")
print("\nOK todos los puntos comparten diseño de observación, partición, "
      "objetivo de evaluación y configuración MCMC.")

# Alias de las cantidades comunes, para no repetir EST[M][...] en cada celda.
T          = EST[next(iter(EST))]["T"]
T0         = EST[next(iter(EST))]["T0"]
N_LAGS     = EST[next(iter(EST))]["n_lags"]
NIVEL      = EST[next(iter(EST))]["nivel"]
MODO_RESIDUO = EST[next(iter(EST))]["modo_residuo"]
OBJETIVO   = EST[next(iter(EST))]["objetivo"]
VENTANAS_W = EST[next(iter(EST))]["ventanas_w"]
W_REF      = VENTANAS_W[len(VENTANAS_W) // 2]
print(f"ventanas w = {VENTANAS_W}   ·   referencia w={W_REF}")
print(f"IC de la curva: NIVEL = 1 - alpha = {NIVEL:.2f}   ->   alpha = {1 - NIVEL:.2f}   (cuantiles F^-1(alpha/2), F^-1(1 - alpha/2))")


In [ ]:
for M in EST:
    e, P = EST[M], EST[M]["paths"]

    fpca = cargar_fpca(P)
    _ver = fpca.verificar()
    assert _ver["todo_ok"], f"[M={M}] las identidades FPCA no se cumplen: {_ver}"

    std = cargar_estandarizador(P, DataStandardizer)
    X_obs, grilla = cargar_curvas(P)          # (T, G) con ruido
    X_true        = cargar_curvas_true(P)     # (T, G) verdadera - solo referencia
    fr, THETA, _  = cargar_representacion(P)
    X_suav        = fr.reconstruct(THETA)     # (T, G) OBJETIVO: curva suavizada

    assert e["component_idx"] == list(range(fpca.M)), (
        "La propagación funcional necesita el vector completo de scores en "
        f"orden; [M={M}] COMPONENT_IDX={e['component_idx']} y M={fpca.M}.")

    e.update({"fpca": fpca, "std": std, "X_obs": X_obs, "X_true": X_true,
              "X_suav": X_suav, "fr": fr,
              "grilla": grilla, "M_fpca": fpca.M, "K": fpca.K,
              "Psi_grid": fpca.Psi_grid, "mu_grid": fpca.mu_grid,
              "SCORES": fpca.SCORES,
              "var_explicada": float(fpca.meta["var_explained"])})

    print(f"M={M}  FPCA M={fpca.M} K={fpca.K}  var. explicada="
          f"{e['var_explicada']:.4%}  ·  curvas {X_true.shape}  "
          f"grilla {grilla.shape}")

# Las curvas y la grilla son las MISMAS en todos los puntos: el generador y la
# semilla no dependen de M, sólo cambia cuántas componentes se retienen. Si no
# coincidieran, el MISE de dos M no sería sobre los mismos datos.
_M0 = next(iter(EST))
for M in EST:
    assert np.allclose(EST[M]["X_true"], EST[_M0]["X_true"]), (
        f"[M={M}] las curvas verdaderas difieren de las de M={_M0}: los puntos "
        "del barrido no salen de la misma simulación.")
    assert np.allclose(EST[M]["grilla"], EST[_M0]["grilla"])
grilla = EST[_M0]["grilla"]
X_true = EST[_M0]["X_true"]
X_obs  = EST[_M0]["X_obs"]
X_suav = EST[_M0]["X_suav"]
print(f"\nOK las {len(EST)} representaciones vienen de la MISMA simulación: "
      "las diferencias entre M son de representación, no de datos.")
print(f"sd(observada - verdadera) = {(X_obs - X_true).std():.4f}   "
      "<- el ruido que el modelo NO debe predecir")

In [ ]:
for M in EST:
    e, P = EST[M], EST[M]["paths"]

    fpca = cargar_fpca(P)
    _ver = fpca.verificar()
    assert _ver["todo_ok"], f"[M={M}] las identidades FPCA no se cumplen: {_ver}"

    std = cargar_estandarizador(P, DataStandardizer)
    X_obs, grilla = cargar_curvas(P)          # (T, G) con ruido
    X_true        = cargar_curvas_true(P)     # (T, G) verdadera - solo referencia
    fr, THETA, _  = cargar_representacion(P)
    X_suav        = fr.reconstruct(THETA)     # (T, G) OBJETIVO: curva suavizada

    assert e["component_idx"] == list(range(fpca.M)), (
        "La propagación funcional necesita el vector completo de scores en "
        f"orden; [M={M}] COMPONENT_IDX={e['component_idx']} y M={fpca.M}.")

    e.update({"fpca": fpca, "std": std, "X_obs": X_obs, "X_true": X_true,
              "X_suav": X_suav, "fr": fr,
              "grilla": grilla, "M_fpca": fpca.M, "K": fpca.K,
              "Psi_grid": fpca.Psi_grid, "mu_grid": fpca.mu_grid,
              "SCORES": fpca.SCORES,
              "var_explicada": float(fpca.meta["var_explained"])})

    print(f"M={M}  FPCA M={fpca.M} K={fpca.K}  var. explicada="
          f"{e['var_explicada']:.4%}  ·  curvas {X_true.shape}  "
          f"grilla {grilla.shape}")

# Las curvas y la grilla son las MISMAS en todos los puntos: el generador y la
# semilla no dependen de M, sólo cambia cuántas componentes se retienen. Si no
# coincidieran, el MISE de dos M no sería sobre los mismos datos.
_M0 = next(iter(EST))
for M in EST:
    assert np.allclose(EST[M]["X_true"], EST[_M0]["X_true"]), (
        f"[M={M}] las curvas verdaderas difieren de las de M={_M0}: los puntos "
        "del barrido no salen de la misma simulación.")
    assert np.allclose(EST[M]["grilla"], EST[_M0]["grilla"])
grilla = EST[_M0]["grilla"]
X_true = EST[_M0]["X_true"]
X_obs  = EST[_M0]["X_obs"]
X_suav = EST[_M0]["X_suav"]
print(f"\nOK las {len(EST)} representaciones vienen de la MISMA simulación: "
      "las diferencias entre M son de representación, no de datos.")
print(f"sd(observada - verdadera) = {(X_obs - X_true).std():.4f}   "
      "<- el ruido que el modelo NO debe predecir")

## 2. Trazas MCMC

Una lectura por punto del barrido. El `feature_names` del `.mat` se verifica
contra las columnas del dataset: con $M$ distintos el número de covariables
cambia, y un `.mat` del punto equivocado se detectaría aquí y no más adelante
como un resultado raro.

In [ ]:
sin_trazas = []

for M in list(EST):
    e, P = EST[M], EST[M]["paths"]
    ci, n_comp, n_iter = e["component_idx"], e["n_components"], e["n_iter"]

    faltan = [ruta_traza(P, ci[k] + 1, c + 1).name
              for k in range(n_comp) for c in range(n_iter)
              if not ruta_traza(P, ci[k] + 1, c + 1).exists()]
    if faltan:
        msg = f"faltan {len(faltan)} trazas: {faltan[:3]}{'...' if len(faltan) > 3 else ''}"
        if not SALTAR_M_SIN_TRAZAS:
            raise AssertionError(f"[M={M}] {msg}\nEjecuta psbp_fd_iteracion.m.")
        sin_trazas.append((M, msg))
        del EST[M]
        continue

    models_chains = {k: {} for k in range(n_comp)}
    for k in range(n_comp):
        esperado = list(e["dfs_train"][k].columns[1:])
        for c in range(n_iter):
            traces, burn, feat = leer_traza(ruta_traza(P, ci[k] + 1, c + 1))
            assert feat == esperado, (
                f"[M={M} k={k} c={c+1}] feature_names != columnas del dataset:\n"
                f"  mat  = {feat}\n  train= {esperado}")
            models_chains[k][c] = ModeloTraza(traces, burn, feat)
    e["models_chains"] = models_chains
    print(f"OK M={M}: {n_comp} x {n_iter} cadenas · {e['n_post']} draws "
          f"posteriores c/u  ·  p={models_chains[0][0].n_features_}")

assert EST, ("Ningún punto del barrido tiene trazas. Ejecuta "
             "psbp_fd_iteracion.m con M_FPCA_LIST antes de este notebook.")
for M, motivo in sin_trazas:
    print(f"! M={M} saltado: {motivo}")

M_OK = tuple(sorted(EST))
print(f"\npuntos evaluables: M = {list(M_OK)}")

## 3. Predicción a $h=1$ sobre la serie completa


In [ ]:
# Número de extracciones de la predictiva por draw posterior. El total es
#     S = (nsim - burn) × S_POR_ITER × n_cadenas
# y d=1 ya da varios miles: subirlo sólo reduce el error Monte Carlo de estimar
# los cuantiles, no cambia la predictiva.
S_POR_ITER = 3
S_FUNC     = 1000        # muestras funcionales tras adelgazar
SEED_PRED  = 20260823
CADA       = 30        # [CONFIG] un extracto de curva cada CADA períodos

# Serie de orígenes: es la MISMA en todos los M (T, T0 y n_lags son comunes).
t_orig   = np.arange(N_LAGS + 1, T + 1)      # tiempo del experimento, base-1
n_orig   = len(t_orig)
T0_orig  = T0 - N_LAGS                       # corte dentro de la serie de orígenes
es_train = t_orig <= T0
print(f"orígenes evaluados: {n_orig}  (train {es_train.sum()} · test {(~es_train).sum()})")

_mem = sum(EST[M]["n_post"] * S_POR_ITER * EST[M]["n_iter"] * n_orig
           * EST[M]["n_components"] * 4 for M in M_OK) / 1e9
print(f"SC_draws de TODO el barrido reservará {_mem * 1e3:.0f} MB en float32")
assert _mem < 2.0, (
    f"SC_draws pediría {_mem:.1f} GB sumando los M. Baja S_POR_ITER o acorta "
    "M_FPCA_LIST antes de continuar.")
print(f"X_draws se libera dentro del bucle: pico ≈ "
      f"{S_FUNC * n_orig * len(grilla) * 8 / 1e6:.0f} MB (un solo M a la vez)")

In [ ]:
_CACHE_PRED = {}

for M in M_OK:
    e = EST[M]
    n_comp, n_iter, n_post = e["n_components"], e["n_iter"], e["n_post"]
    ci = e["component_idx"]

    dfs_full = {k: pd.concat([e["dfs_train"][k], e["dfs_test"][k]],
                             ignore_index=True) for k in range(n_comp)}
    assert len(dfs_full[0]) == n_orig, f"[M={M}] {len(dfs_full[0])} != {n_orig}"

    # -- momentos y extracciones por score, agrupando cadenas ----------------
    #    momentos : ley de varianza total entre cadenas (fit.agrupar_momentos)
    #    muestras : concatenación, que es la mezcla de igual peso
    Y_obs = np.column_stack([dfs_full[k].iloc[:, 0].to_numpy()
                             for k in range(n_comp)])
    Y_hat = np.empty_like(Y_obs)
    Y_sd  = np.empty_like(Y_obs)

    S_POR_CADENA = n_post * S_POR_ITER
    S_total      = S_POR_CADENA * n_iter
    SC_draws     = np.empty((S_total, n_orig, n_comp), dtype=np.float32)

    for k in range(n_comp):
        medias, sds = [], []
        for j, c in enumerate(sorted(e["models_chains"][k])):
            if (ci[k], c) not in _CACHE_PRED:
                _mom = e["models_chains"][k][c].momentos(dfs_full[k])
                _mue = e["models_chains"][k][c].muestrear(
                    dfs_full[k], S_POR_ITER, seed=SEED_PRED + 1000 * ci[k] + c)
                _CACHE_PRED[(ci[k], c)] = (_mom["media"], _mom["sd"], _mue)
            _m, _s, muestras_c = _CACHE_PRED[(ci[k], c)]
            medias.append(_m)
            sds.append(_s)          # PREDICTIVA (v3), no la del centro
            assert muestras_c.shape == (S_POR_CADENA, n_orig), (
                f"[M={M} k={k} c={c}] muestrear devolvió {muestras_c.shape}; se "
                f"esperaba ({S_POR_CADENA}, {n_orig}).")
            SC_draws[j * S_POR_CADENA:(j + 1) * S_POR_CADENA, :, k] = muestras_c
            del muestras_c
        Y_hat[:, k], Y_sd[:, k] = agrupar_momentos(np.column_stack(medias),
                                                   np.column_stack(sds))

    li_s, ls_s = intervalo_muestral(SC_draws, nivel=NIVEL)     # (n_orig, M)

    # -- propagación a la curva ---------------------------------------------
    paso_thin = max(1, S_total // S_FUNC)
    propagador = PropagadorFuncional(e["Psi_grid"], e["mu_grid"],
                                     estandarizador=e["std"],
                                     modo_residuo=MODO_RESIDUO)
    X_draws = propagador.curvas_desde_scores(SC_draws[::paso_thin], seed=SEED_PRED)
    X_pred  = X_draws.mean(axis=0)                              # (n, G)
    li_f = np.quantile(X_draws, (1 - NIVEL) / 2, axis=0)
    ls_f = np.quantile(X_draws, 1 - (1 - NIVEL) / 2, axis=0)

    X_obj_ev  = X_suav[N_LAGS:]      # (n_orig, G) OBJETIVO: curva suavizada
    X_obs_ev  = X_obs[N_LAGS:]       # (n_orig, G) observada, sólo para las figuras
    X_proj    = e["fpca"].reconstruct(e["SCORES"])[N_LAGS:]     # piso de la representación

    S_funcional = int(X_draws.shape[0])
    del X_draws

    e.update({"dfs_full": dfs_full, "Y_obs": Y_obs, "Y_hat": Y_hat, "Y_sd": Y_sd,
              "SC_draws": SC_draws, "li_s": li_s, "ls_s": ls_s,
              "X_pred": X_pred, "li_f": li_f, "ls_f": ls_f, "X_proj": X_proj,
              "S_total": int(S_total),
              "S_funcional": S_funcional, "paso_thin": paso_thin})

    print(f"M={M}: S={S_total} extracciones por score · funcionales "
          f"{S_funcional} (1 de cada {paso_thin}) · SC_draws "
          f"{SC_draws.nbytes / 1e6:.0f} MB")
    print(f"       MISE del TRUNCAMIENTO FPCA (test) = "
          f"{mise(X_obj_ev[~es_train], X_proj[~es_train], grilla):.6f}"
          "   <- distancia del objetivo a su propia proyección en M")

X_obj_ev  = X_suav[N_LAGS:]
X_obs_ev  = X_obs[N_LAGS:]

### 3.1 Persistencia

La banda y la predicción puntual se guardan en `predict/` -no en `reports/`,
es un objeto de datos y no una salida de lectura- porque `101_05` las necesita
para calcular el Winkler del PSBPM-FD sobre su propia ventana móvil sin volver
a muestrear la predictiva completa.

In [ ]:

for M in M_OK:
    e = EST[M]
    destino = e["paths"]["predict"] / "banda_funcional_psbp.npz"
    np.savez_compressed(
        destino,
        li=e["li_f"].astype(np.float32), ls=e["ls_f"].astype(np.float32),
        X_pred=e["X_pred"].astype(np.float32),
        t_orig=t_orig, nivel=np.array([NIVEL]),
        n_lags=np.array([N_LAGS]), T0=np.array([T0]),
        modo_residuo=np.array([MODO_RESIDUO]),
        objetivo=np.array([OBJETIVO]))
    print(f"[M={M}] banda funcional -> {destino.name}  "
          f"({destino.stat().st_size / 1e6:.1f} MB, {e['li_f'].shape})")

print("\nEs la banda por CUANTILES de la predictiva muestral: no supone forma")
print("alguna, y es la que el 101_05 contrasta contra la banda gaussiana del")
print("modelo AR.")


## 4. Bloque A: error puntual sobre la ventana móvil

Las tres normas del MISMO error $e_t(\tau) = X_t(\tau) - \hat X_t(\tau)$:

| | qué estima | qué añade |
|---|---|---|
| **MAE** ($L^1$) | mediana condicional | robusta: un origen catastrófico no la mueve |
| **RMSE** ($L^2$) | media condicional | la que usa el PSBPM-FD y el resto del capítulo |
| **Error máximo** ($L^\infty$, promedio por curva) | peor caso | por curva, $\max_\tau \lvert e_t(\tau)\rvert$; sobre la ventana, el promedio de esos $w$ máximos |

Todo se calcula UNA vez por M en `tablas_fun` -la misma tabla que consume el
Bloque B de §5- y se dibuja aquí SOLO el Bloque A.

In [ ]:
for M in M_OK:
    e = EST[M]
    def _tablas(objetivo, verboso):
        return {w: ventana_movil_funcional(objetivo, e["X_pred"], grilla, T0_orig,
                                           w=w, li=e["li_f"], ls=e["ls_f"],
                                           t_offset=N_LAGS, pesos_tau=PESOS_TAU,
                                           nivel=NIVEL, bloque_A=True,
                                           verbose=(verboso and w == W_REF))
                for w in VENTANAS_W}

    # Los dos objetivos de docs 03_05_00: la curva suavizada y X_t^(M).
    tablas_fun = _tablas(X_obj_ev, True)
    e["tablas_fun"] = tablas_fun
    e["tablas_rep"] = _tablas(e["X_proj"], False)
    tablas_fun[W_REF].to_csv(
        e["paths"]["out_report"] / f"57_ventana_funcional_w{W_REF}.csv", index=False)

In [ ]:
# Una figura por ancho con los M superpuestos (la plantilla hacia lo inverso).
def apilar_por_w(w, columnas):
    """Tablas de todos los M en formato largo, con el punto como grupo."""
    partes = []
    for M in M_OK:
        t = EST[M]["tablas_fun"][w].copy()
        faltan = [c for c in columnas if c not in t.columns]
        assert not faltan, f"[M={M}, w={w}] faltan columnas: {faltan}"
        t.insert(0, "punto", f"M={M}")
        partes.append(t)
    out = pd.concat(partes, ignore_index=True)
    out.attrs["w"] = int(w)
    return out


TABLAS_W = {w: apilar_por_w(w, METRICAS_A_COLS + METRICAS_B_COLS)
            for w in VENTANAS_W}

for w in VENTANAS_W:
    plot_ventana_movil(
        TABLAS_W[w], T0, METRICAS_A_COLS, columna_grupo="punto",
        title=f"Bloque A · ventana w={w} · todos los M · "
              "MAE, RMSE, error máximo",
        save_path=str(PATH_BARRIDO / f"85_bloqueA_w{w}.png"),
        verbose=(w == W_REF))
    plt.show()

# La cadena va sobre l2_medio, no sobre rmse_f: rmse_f es media CUADRATICA
# entre origenes y puede superar a linf_medio si el error es desigual.
for M in M_OK:
    for w in VENTANAS_W:
        tfun = EST[M]["tablas_fun"][w]
        assert (tfun["mae_f"] <= tfun["l2_medio"] + 1e-9).all(), (
            f"[M={M}, w={w}] mae_f > l2_medio: revisar la cuadratura.")
        assert (tfun["l2_medio"] <= tfun["linf_medio"] + 1e-9).all()
        assert (tfun["linf_medio"] <= tfun["linf_max"] + 1e-9).all()
print(f"cadena L^1 <= L^2 <= L^inf verificada en los {len(M_OK)} puntos "
      f"x {len(VENTANAS_W)} anchos")


### 4.1 ¿Qué $M$ gana en cada ventana, y en cada ancho?


In [ ]:
# Las ventanas de todos los M tienen que coincidir posicion a posicion.
for w in VENTANAS_W:
    _ref_w = EST[M_OK[0]]["tablas_fun"][w]
    for M in M_OK[1:]:
        assert list(EST[M]["tablas_fun"][w]["t_centro"]) == list(_ref_w["t_centro"]), (
            f"[M={M}, w={w}] las ventanas no estan alineadas con M={M_OK[0]}: "
            "revisar que T, T0, n_lags y VENTANAS_W sean comunes.")


def ganancias_por_w(w, metricas):
    """% de ventanas ganadas por cada M, por metrica y bloque, en el ancho w."""
    ref = EST[M_OK[0]]["tablas_fun"][w]
    filas = []
    for etiqueta, col in metricas:
        piv = pd.DataFrame({M: EST[M]["tablas_fun"][w][col].to_numpy()
                            for M in M_OK})
        # En picp/picp_simultaneo el objetivo es el NOMINAL, no el minimo.
        objetivo = (piv - NIVEL).abs() if col.startswith("picp") else piv
        tabla_g = pd.DataFrame({
            "t_centro": ref["t_centro"], "bloque": ref["bloque"],
            "cruza_T0": ref["cruza_T0"], "ganador": objetivo.idxmin(axis=1)})
        limpio_g = tabla_g[~tabla_g["cruza_T0"]]
        for bloque in ("train", "test"):
            sub_b = limpio_g[limpio_g.bloque == bloque]
            n_tot = len(sub_b)
            for M in M_OK:
                filas.append({
                    "w": int(w), "metrica": etiqueta, "columna": col,
                    "criterio": ("|valor - nominal|" if col.startswith("picp")
                                 else "menor es mejor"),
                    "bloque": bloque, "M": M, "n_ventanas": n_tot,
                    "pct_ventanas_ganadas": (float((sub_b["ganador"] == M).mean())
                                             if n_tot else float("nan"))})
    return pd.DataFrame(filas)


def figura_ganador(gana_df, metricas, w, titulo, destino):
    fig, axes = plt.subplots(1, len(metricas), figsize=(4.3 * len(metricas), 4.2),
                             sharey=True, squeeze=False)
    axes = axes[0]
    ancho_barra = 0.8 / 2
    for ax, (etiqueta, _) in zip(axes, metricas):
        sub_m = gana_df[(gana_df.metrica == etiqueta) & (gana_df.w == w)]
        xs = np.arange(len(M_OK))
        for i_b, bloque in enumerate(("train", "test")):
            vals = [float(sub_m[(sub_m.M == M) & (sub_m.bloque == bloque)]
                          ["pct_ventanas_ganadas"].iloc[0]) for M in M_OK]
            ax.bar(xs + (i_b - 0.5) * ancho_barra, vals, width=ancho_barra,
                   label=bloque,
                   color=("#2c7fb8" if bloque == "train" else "#c0392b"))
        ax.set_xticks(xs); ax.set_xticklabels([f"M={M}" for M in M_OK])
        ax.set_title(etiqueta, fontsize=9)
        ax.set_ylim(0, 1); ax.legend(fontsize=8)
    axes[0].set_ylabel("% de ventanas ganadas")
    fig.suptitle(titulo, fontsize=12)
    fig.tight_layout()
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.show()


GANA_A = pd.concat([ganancias_por_w(w, METRICAS_A) for w in VENTANAS_W],
                   ignore_index=True)
GANA_B = pd.concat([ganancias_por_w(w, METRICAS_B) for w in VENTANAS_W],
                   ignore_index=True)
GANA_A.to_csv(PATH_BARRIDO / "84_ganador_por_ventana.csv", index=False)
GANA_B.to_csv(PATH_BARRIDO / "87_ganador_bloqueB.csv", index=False)

for w in VENTANAS_W:
    figura_ganador(GANA_A, METRICAS_A, w,
                   f"{EXPERIMENT_BASE} · Bloque A · qué M gana cada ventana (w={w})",
                   PATH_BARRIDO / f"84_ganador_por_ventana_w{w}.png")

display(GANA_A[GANA_A.bloque == "test"]
        .pivot_table(index=["metrica", "w"], columns="M",
                     values="pct_ventanas_ganadas")
        .style.format("{:.1%}")
        .set_caption("Bloque A · % de ventanas TEST ganadas por M, en cada "
                     "ancho (84_ganador_por_ventana.csv)"))


In [ ]:
# Lectura numérica del salto en T0, excluyendo las ventanas que lo cruzan.
for M in M_OK:
    e = EST[M]
    print(f"── M={M} ──")
    print(f"{'w':>4}  {'MISE train':>11}  {'MISE test':>11}  {'salto':>7}")
    print(f"{'-'*4}  {'-'*11}  {'-'*11}  {'-'*7}")
    saltos = {}
    for w in VENTANAS_W:
        t_ = e["tablas_fun"][w]
        limpio = t_[~t_["cruza_T0"]]
        a = limpio.loc[limpio.bloque == "train", "mise"].mean()
        b = limpio.loc[limpio.bloque == "test",  "mise"].mean()
        saltos[w] = float(b / a)
        print(f"{w:>4}  {a:>11.6f}  {b:>11.6f}  {b/a:>6.2f}x")
    e["saltos_T0"] = saltos

## 5. Bloque B: intervalos de predicción, en una sola vista

Cuatro cifras del intervalo, ninguna "primaria" en esta vista -el Winkler es
la única regla de puntuación propia y por eso es la que ordena modelos cuando
hace falta ordenar, pero aquí se muestran las cuatro juntas-:

| | qué es | de qué es |
|---|---|---|
| **Winkler** | interval score | penaliza ancho Y cobertura a la vez |
| **Cobertura simultánea** | de $w$ curvas, cuántas quedan COMPLETAS dentro de su banda (1/0 por curva) | fracción de CURVAS |
| **Cobertura puntual (PICP)** | de una curva, qué fracción de $\tau$ cae dentro | promedio de esa fracción sobre la ventana |
| **MPIW** | ancho medio de la banda | — |

La cobertura simultánea es SIEMPRE $\le$ la puntual: basta un $\tau$ fuera
para que la curva entera cuente como no cubierta.

In [ ]:
# Una figura por ancho con los M superpuestos; TABLAS_W viene de la seccion 4.
for w in VENTANAS_W:
    plot_ventana_movil(
        TABLAS_W[w], T0, METRICAS_B_COLS, columna_grupo="punto",
        title=f"Bloque B · ventana w={w} · todos los M · "
              "Winkler, cobertura simultánea, PICP, MPIW",
        save_path=str(PATH_BARRIDO / f"86_bloqueB_w{w}.png"),
        verbose=(w == W_REF))
    plt.show()

for M in M_OK:
    for w in VENTANAS_W:
        tfun = EST[M]["tablas_fun"][w]
        assert (tfun["picp_simultaneo"] <= tfun["picp"] + 1e-9).all(), (
            f"[M={M}, w={w}] la cobertura simultanea supero a la puntual: no "
            "puede pasar por definicion.")

# Ventanas ganadas del Bloque B (en picp, contra el nominal y no el minimo).
for w in VENTANAS_W:
    figura_ganador(GANA_B, METRICAS_B, w,
                   f"{EXPERIMENT_BASE} · Bloque B · qué M gana cada ventana (w={w})",
                   PATH_BARRIDO / f"87_ganador_bloqueB_w{w}.png")

display(GANA_B[GANA_B.bloque == "test"]
        .pivot_table(index=["metrica", "w"], columns="M",
                     values="pct_ventanas_ganadas")
        .style.format("{:.1%}")
        .set_caption("Bloque B · % de ventanas TEST ganadas por M, en cada "
                     "ancho (87_ganador_bloqueB.csv)"))


## 6. Muestra de predicciones

Dos vistas: un extracto regular del bloque de **prueba** (para no mezclar con
in-sample), y las ventanas con **peor** valor de `METRICA_PEORES` ([CONFIG]).
En la segunda se agrega la curva **anterior** ($t-1$, gris punteada): es el
primer diagnóstico de POR QUÉ el modelo se equivocó -¿la curva de partida ya
era atípica?-.

In [ ]:
# Extractos regulares, SOLO del bloque de prueba.
mask_test = ~es_train
for M in M_OK:
    e = EST[M]
    plot_extractos_curvas(
        X_obj_ev[mask_test], e["X_pred"][mask_test], e["li_f"][mask_test],
        e["ls_f"][mask_test], grilla, T0, t=t_orig[mask_test],
        cada=CADA, n_col=5, nivel=NIVEL,
        etiqueta_objetivo="curva suavizada (objetivo)", X_obs=X_obs_ev[mask_test],
        title=f"M={M} - predictiva funcional y banda de credibilidad (test)",
        save_path=str(e["paths"]["out_report"] / "54_extractos_curvas_test.png"),
        verbose=True)
    plt.show()


In [ ]:
# Zoom sobre la frontera: los últimos orígenes de train y los primeros de test.
# Es donde se ve, curva a curva, si la banda se ensancha al salir de muestra.
i_corte = int(np.searchsorted(t_orig, T0))
sel = np.arange(max(0, i_corte - 3), min(n_orig, i_corte + 4))

for M in M_OK:
    e = EST[M]
    plot_extractos_curvas(
        X_obj_ev[sel], e["X_pred"][sel], e["li_f"][sel], e["ls_f"][sel],
        grilla, T0, t=t_orig[sel], cada=1, n_col=len(sel), nivel=NIVEL,
        etiqueta_objetivo="curva suavizada (objetivo)", X_obs=X_obs_ev[sel],
        title=f"M={M} - frontera train/test (T0={T0})",
        save_path=str(e["paths"]["out_report"] / "55_extractos_frontera.png"),
        verbose=True)
    plt.show()

In [ ]:
X_prev_full = np.full_like(X_obj_ev, np.nan)
X_prev_full[1:] = X_obj_ev[:-1]

for M in M_OK:
    e = EST[M]
    tfun = e["tablas_fun"][W_REF]
    candidatas = tfun[(tfun["bloque"] == "test") & (~tfun["cruza_T0"])]
    if METRICA_PEORES not in candidatas.columns:
        print(f"[M={M}] {METRICA_PEORES!r} no esta en la tabla de ventana; "
              "revisa METRICA_PEORES en [CONFIG].")
        continue
    peores = candidatas.nlargest(N_PEORES, METRICA_PEORES)

    print(f"[M={M}] peores {len(peores)} ventanas por {METRICA_PEORES} "
          f"(w={W_REF}, sólo test, sin cruzar T0):")
    for _, row in peores.iterrows():
        print(f"    t=[{int(row['t_ini'])}, {int(row['t_fin'])}]  "
              f"{METRICA_PEORES}={row[METRICA_PEORES]:.6f}")

    for _, row in peores.iterrows():
        t_ini_exp, t_fin_exp = int(row["t_ini"]), int(row["t_fin"])
        sel = np.where((t_orig >= t_ini_exp) & (t_orig <= t_fin_exp))[0]
        if sel.size == 0:
            continue
        plot_extractos_curvas(
            X_obj_ev[sel], e["X_pred"][sel], e["li_f"][sel], e["ls_f"][sel],
            grilla, T0, t=t_orig[sel], cada=1, n_col=min(len(sel), 7),
            nivel=NIVEL, etiqueta_objetivo="curva suavizada (objetivo)",
            X_obs=X_obs_ev[sel], X_prev=X_prev_full[sel],
            title=f"M={M} · peor ventana por {METRICA_PEORES} "
                  f"(t=[{t_ini_exp},{t_fin_exp}], valor={row[METRICA_PEORES]:.4f})",
            save_path=str(e["paths"]["out_report"]
                          / f"56b_peor_ventana_{METRICA_PEORES}_t{t_ini_exp}.png"),
            verbose=False)
        plt.show()


## 7. Probabilidades posteriores de inclusión (PIP)

Contra la estructura del generador. En el Algoritmo A-1 la curva $t$ es el
bloque **contiguo** al $t-1$ dentro de una única trayectoria escalar, de modo
que todo el bloque anterior influye sobre todo el actual y, proyectado sobre la
base FPCA, **toda componente rezagada es en principio activa**: no hay aquí un
conjunto activo pequeño que contrastar con especificidad/AUC, como sí lo hay en
escenarios con estructura dispersa. La cifra útil es `pip_media_activas`.


In [ ]:
for M in M_OK:
    e = EST[M]
    pip_df = matriz_pip(e["models_chains"], e["burn"],
                        component_idx=e["component_idx"], verbose=True)
    pip_df.to_csv(e["paths"]["out_report"] / "46_pip.csv")
    e["pip_df"] = pip_df

    cols_pip = [c for c in pip_df.columns if not c.endswith("_sd")]
    display(pip_df.style
        .background_gradient(subset=cols_pip, cmap="RdYlGn", vmin=0, vmax=1)
        .format("{:.3f}")
        .set_caption(f"M={M} · P(gamma_j = 1 | datos), inclusión global, "
                     "media entre cadenas ± sd"))

    sd_cols = [c for c in pip_df.columns if c.endswith("_sd")]
    sd_max = float(pip_df[sd_cols].to_numpy().max()) if sd_cols else float("nan")
    e["pip_sd_max"] = sd_max
    print(f"[M={M}] dispersión máxima entre cadenas: {sd_max:.3f}"
          + ("   ! las cadenas discrepan sobre la selección" if sd_max > 0.15
             else "   OK las cadenas coinciden"))


In [ ]:
for M in M_OK:
    e = EST[M]
    # [CONFIG] estructura verdadera del generador. Escenario A-1: el bloque t
    # es contiguo al t-1 en una unica trayectoria escalar => todas las
    # covariables son potencialmente activas.
    VERDAD = {f"FPC {e['component_idx'][k] + 1}": list(e["cov_por_componente"][k])
              for k in range(e["n_components"])}

    contraste = contraste_con_verdad(e["pip_df"], VERDAD, umbral=0.5)
    e["contraste"] = contraste
    if not contraste.empty:
        contraste.to_csv(e["paths"]["out_report"] / "47_contraste_pip.csv")
        display(contraste.style.format("{:.3f}", subset=[
            c for c in contraste.columns if contraste[c].dtype.kind == "f"])
            .set_caption(f"M={M} · selección de variables vs estructura del "
                         "generador (umbral 0.5)"))

print("Con todas las covariables activas, especificidad y AUC no están "
      "definidas de forma informativa: la cifra útil es `pip_media_activas`.")


## 8. Cuadro resumen: mínimo, máximo y promedio

La ventana móvil genera una SERIE por métrica, no una cifra. Esta tabla es la
lectura resumida de esa serie: mínimo, máximo y promedio, por bloque
train/test, excluyendo las ventanas que cruzan $T_0$ -su cifra mezcla dentro y
fuera de muestra-. Cubre las 7 métricas de los Bloques A y B; ninguna se
destaca sobre las demás.

In [ ]:
METRICAS_RESUMEN = METRICAS_A + METRICAS_B   # las 6 con lectura por ventana

# Los dos objetivos en la misma tabla, con la columna que declara cual.
OBJETIVOS = (("curva_suavizada", "tablas_fun"), ("representacion_fpca", "tablas_rep"))

resumen_M = []
for M in M_OK:
    e = EST[M]
    limpio = e["tablas_fun"][W_REF][~e["tablas_fun"][W_REF]["cruza_T0"]]

    filas = []
    for objetivo, clave in OBJETIVOS:
        tab = e[clave][W_REF]
        lim = tab[~tab["cruza_T0"]]
        for etiqueta, col in METRICAS_RESUMEN:
            for bloque in ("train", "test"):
                v = lim.loc[lim.bloque == bloque, col]
                filas.append({"objetivo": objetivo, "metrica": etiqueta,
                              "columna": col, "bloque": bloque,
                              "minimo": float(v.min()), "maximo": float(v.max()),
                              "promedio": float(v.mean())})
    resumen = pd.DataFrame(filas).set_index(["objetivo", "metrica", "bloque"])
    resumen.to_csv(e["paths"]["out_report"] / "66_metricas_resumen.csv")
    e["resumen_df"] = resumen

    display(resumen.style
            .format({"minimo": "{:.6f}", "maximo": "{:.6f}", "promedio": "{:.6f}"})
            .set_caption(f"M={M} · mínimo / máximo / promedio por ventana "
                         f"(w={W_REF}, sin cruzar T0) · nivel funcional"))

    for objetivo, clave in OBJETIVOS:
        tab = e[clave][W_REF]
        lim = tab[~tab["cruza_T0"]]
        picp_pt = float(lim.loc[lim.bloque == "test", "picp"].mean())
        picp_si = float(lim.loc[lim.bloque == "test", "picp_simultaneo"].mean())
        print(f"[M={M}] {objetivo:<20} PICP puntual test = {picp_pt:.4f} "
              f"(nominal {NIVEL:.2f})  ·  PICP simultáneo test = {picp_si:.4f}")

    # El indicador I_t por origen, en sus dos versiones -de las que (picp) y
    # (picp_simultaneo) son el promedio-. Se persiste porque el promedio
    # esconde DONDE falla la banda.
    partes_ind = []
    for objetivo, Y in (("curva_suavizada", X_obj_ev),
                        ("representacion_fpca", e["X_proj"])):
        I_t_puntual    = indicador_cobertura(Y, e["li_f"], e["ls_f"]).mean(axis=1)
        I_t_simultaneo = indicador_cobertura_simultanea(Y, e["li_f"], e["ls_f"])
        partes_ind.append(pd.DataFrame({
            "objetivo": objetivo,
            "t": np.arange(len(I_t_puntual)) + N_LAGS + 1,
            "bloque": np.where(es_train, "train", "test"),
            "I_t_puntual": I_t_puntual,
            "I_t_simultaneo": I_t_simultaneo.astype(float),
        }))
    ind = pd.concat(partes_ind, ignore_index=True)
    ind.to_csv(e["paths"]["out_report"] / "67_indicador_cobertura.csv",
               index=False)

    m_ = resumen.reset_index()
    m_["M"] = M
    resumen_M.append(m_)

resumen_barrido = pd.concat(resumen_M, ignore_index=True)
resumen_barrido.to_csv(PATH_BARRIDO / "89_resumen_por_M.csv", index=False)
display(resumen_barrido[resumen_barrido.bloque == "test"]
        .pivot(index="metrica", columns="M", values="promedio")
        .style.format("{:.6f}")
        .set_caption("Promedio en TEST contra M (89_resumen_por_M.csv)"))


## 9. Monitoreo univariado por componente FPCA

Cada componente FPCA retenida se predice de manera univariada dentro del
modelo -es una mezcla por componente-, así que aquí se mira su MAE, RMSE y
MISE (= RMSE² del score, el análogo puntual del MISE funcional) por separado,
sobre la ventana móvil. Con $M=1$ hay una sola línea por panel; con $M=k$ hay
$k$ líneas, una por componente. **Solo monitoreo**: ninguna de las tres es
primaria.

In [ ]:
# `verbose` sólo en el ancho de referencia: con todos los anchos y todos los M
# la salida serían miles de líneas.
for M in M_OK:
    e = EST[M]
    etiquetas = [f"FPC {i + 1}" for i in e["component_idx"]]
    tablas_score = {
        w: ventana_movil_scores(
            e["Y_obs"], e["Y_hat"], T0_orig, w=w, muestras=e["SC_draws"],
            li=e["li_s"], ls=e["ls_s"], t_offset=N_LAGS,
            etiquetas=etiquetas, verbose=(w == W_REF))
        for w in VENTANAS_W
    }
    e["tablas_score"] = tablas_score
    for w in VENTANAS_W:
        tablas_score[w].to_csv(
            e["paths"]["out_report"] / f"90_ventana_scores_w{w}.csv", index=False)


In [ ]:
# Una figura por ancho; el grupo aqui es la componente FPCA.
for M in M_OK:
    for w in VENTANAS_W:
        tsc = EST[M]["tablas_score"][w].copy()
        tsc["mise"] = tsc["rmse"] ** 2   # MSE del score: analogo puntual del MISE
        plot_ventana_movil(
            tsc, T0, ["mae", "rmse", "mise"], columna_grupo="componente",
            title=f"M={M} · monitoreo univariado por componente · ventana w={w} · "
                  "MAE, RMSE, MISE del score",
            save_path=str(EST[M]["paths"]["out_report"]
                          / f"91_ventana_univariado_w{w}.png"),
            verbose=(w == W_REF))
        plt.show()


## 10. Evaluación puntual de los scores (coeficientes FPCA)

Las secciones anteriores miran la **curva**, que es el producto final. Ésta
mira el objeto que el muestreador predice de verdad: el score $\xi_{t,k}$ de
cada componente. La curva sale de propagarlos, así que un problema de
calibración que aquí se vea localizado en una componente queda, allá,
promediado con las demás y es mucho más difícil de atribuir.

| | qué muestra |
|---|---|
| **§10.1** | el intervalo de credibilidad de cada $\xi_k$ a lo largo del tiempo, con $\xi$ observado encima |
| **§10.2** | las métricas de ese intervalo: PICP, MPIW, Winkler, ACE — globales por bloque y sobre la ventana móvil, un panel por ancho |
| **§10.3** | dispersión $\hat\xi$ contra $\xi$ con la diagonal, componente a componente |

La banda es `li_s`/`ls_s`, los cuantiles muestrales de la predictiva por score
que §3 ya calculó: es **la misma** de la que sale la banda funcional, antes de
propagarla. No se vuelve a muestrear nada.


### 10.1 Cómo se mueve el intervalo de credibilidad de cada score

In [ ]:
# Serie de cada score con su banda de credibilidad; la vertical es T0.
for M in M_OK:
    e = EST[M]
    n_comp = e["n_components"]
    fig, axes = plt.subplots(n_comp, 1, sharex=True,
                             figsize=(12, 2.4 * n_comp), squeeze=False)
    axes = axes[:, 0]
    for k, ax in enumerate(axes):
        ax.fill_between(t_orig, e["li_s"][:, k], e["ls_s"][:, k],
                        color="#2c7fb8", alpha=0.25,
                        label=f"IC {NIVEL:.0%} de la predictiva")
        ax.plot(t_orig, e["Y_obs"][:, k], lw=0.9, color="0.25", label=r"$\xi$ observado")
        ax.plot(t_orig, e["Y_hat"][:, k], lw=1.0, color="#c0392b",
                label=r"$\hat\xi$ (media predictiva)")
        ax.axvline(T0, color="crimson", ls="--", lw=1.2)
        ax.axhline(0, color="0.7", lw=0.6)
        dentro = ((e["Y_obs"][:, k] >= e["li_s"][:, k])
                  & (e["Y_obs"][:, k] <= e["ls_s"][:, k]))
        ax.set_ylabel(rf"$\xi_{{{e['component_idx'][k] + 1}}}$")
        ax.set_title(
            f"FPC {e['component_idx'][k] + 1}   ·   PICP train "
            f"{dentro[es_train].mean():.3f} / test {dentro[~es_train].mean():.3f} "
            f"(nominal {NIVEL:.2f})   ·   ancho medio train "
            f"{(e['ls_s'][es_train, k] - e['li_s'][es_train, k]).mean():.4f} / test "
            f"{(e['ls_s'][~es_train, k] - e['li_s'][~es_train, k]).mean():.4f}",
            fontsize=9, loc="left")
    axes[0].legend(fontsize=8, loc="upper right", ncol=3)
    axes[-1].set_xlabel("t")
    fig.suptitle(f"M={M} · banda de credibilidad de cada score "
                 f"(T0={T0}, nivel {NIVEL:.0%})", fontsize=12)
    fig.tight_layout()
    fig.savefig(e["paths"]["out_report"] / "92_scores_banda_credibilidad.png",
                dpi=150, bbox_inches="tight")
    plt.show()


### 10.2 Métricas de intervalo sobre los scores

Las mismas cuatro cifras del Bloque B, pero por componente y en la escala del
score: **PICP** (fracción de orígenes cubiertos), **MPIW** (ancho medio),
**Winkler** (regla de puntuación que penaliza ancho y fallo a la vez) y
**ACE** = PICP − nominal, que es el signo de la miscalibración: negativo es
subcobertura.

Primero la lectura global por bloque; después la evolución sobre la ventana
móvil, **un panel por ancho**, con el mismo criterio de §4 y §5.


In [ ]:
from model_psbp_fd.fit import winkler as winkler_scores

filas_int = []
for M in M_OK:
    e = EST[M]
    for k in range(e["n_components"]):
        y = e["Y_obs"][:, k]
        lo, hi = e["li_s"][:, k], e["ls_s"][:, k]
        w_t = winkler_scores(y, lo, hi, nivel=NIVEL)
        dentro = (y >= lo) & (y <= hi)
        for bloque, m in (("train", es_train), ("test", ~es_train)):
            filas_int.append({
                "M": M, "componente": f"FPC {e['component_idx'][k] + 1}",
                "bloque": bloque,
                "picp": float(dentro[m].mean()),
                "ace": float(dentro[m].mean() - NIVEL),
                "mpiw": float((hi[m] - lo[m]).mean()),
                "winkler": float(w_t[m].mean()),
                # sd implicada por la banda. No se usa momentos()["sd"]:
                # con atau <= 1 los atomos vacios dejan E[1/tau] sin definir y
                # esa cifra diverge, mientras los cuantiles no se mueven.
                "sd_banda": float((hi[m] - lo[m]).mean() / (2 * 1.959964)),
                "rmse": float(np.sqrt(((y[m] - e["Y_hat"][m, k]) ** 2).mean())),
                # Una banda gaussiana calibrada daria ~3.92.
                "mpiw_sobre_rmse": float(
                    (hi[m] - lo[m]).mean()
                    / max(float(np.sqrt(((y[m] - e["Y_hat"][m, k]) ** 2).mean())), 1e-12)),
            })

INT_SCORES = pd.DataFrame(filas_int)
INT_SCORES.to_csv(PATH_BARRIDO / "88_scores_intervalo_por_M.csv", index=False)
for M in M_OK:
    (INT_SCORES[INT_SCORES.M == M]
     .to_csv(EST[M]["paths"]["out_report"] / "93_scores_intervalo_resumen.csv",
             index=False))

display(INT_SCORES.set_index(["M", "componente", "bloque"])
        .style.format({"picp": "{:.4f}", "ace": "{:+.4f}", "mpiw": "{:.4f}",
                       "winkler": "{:.4f}", "sd_banda": "{:.4f}",
                       "rmse": "{:.4f}", "mpiw_sobre_rmse": "{:.2f}"})
        .background_gradient(subset=["ace"], cmap="RdYlGn", vmin=-0.5, vmax=0.5)
        .set_caption(f"Intervalos de credibilidad de los scores, nivel "
                     f"{NIVEL:.0%} · ACE = PICP - nominal · "
                     "mpiw/rmse ≈ 3.92 si la banda fuera gaussiana y calibrada "
                     "(88_scores_intervalo_por_M.csv)"))

_mala = INT_SCORES[(INT_SCORES.bloque == "test") & (INT_SCORES.ace.abs() > 0.10)]
if len(_mala):
    print("\n! componentes con |ACE| > 0.10 en test (miscalibración que el "
          "promedio funcional del Bloque B esconde):")
    for _, r in _mala.iterrows():
        print(f"    M={r['M']}  {r['componente']}  PICP={r['picp']:.3f}  "
              f"ACE={r['ace']:+.3f}")
else:
    print("\nOK ninguna componente se aleja más de 0.10 del nominal en test.")


In [ ]:
# Evolucion del intervalo sobre la ventana movil, una figura por ancho.
COLS_INT = ["winkler", "picp", "mpiw"]

for M in M_OK:
    e = EST[M]
    for w in VENTANAS_W:
        t_int = e["tablas_score"][w]
        faltan = [c for c in COLS_INT if c not in t_int.columns]
        assert not faltan, (
            f"[M={M}, w={w}] faltan {faltan} en tablas_score: revisar que "
            "ventana_movil_scores se haya llamado con li y ls.")
        plot_ventana_movil(
            t_int, T0, COLS_INT, columna_grupo="componente",
            title=f"M={M} · intervalo de los scores · ventana w={w} · "
                  f"Winkler, PICP (nominal {NIVEL:.2f}), MPIW",
            save_path=str(e["paths"]["out_report"]
                          / f"93_ventana_scores_intervalo_w{w}.png"),
            verbose=(w == W_REF))
        plt.show()


### 10.3 Dispersión $\hat\xi$ contra $\xi$

Un panel por componente, con la diagonal $y=x$. Tres cosas se leen de golpe:

- **la nube pegada a la diagonal** es predicción; la nube **horizontal**
  —$\hat\xi$ casi constante mientras $\xi$ varía— es el modelo prediciendo la
  media incondicional, que es lo que pasa cuando la componente no tiene
  dinámica que capturar;
- **la pendiente** de la recta ajustada: menor que 1 es encogimiento hacia la
  media, que es el comportamiento esperado de un predictor bayesiano y no un
  defecto;
- **las barras** son el intervalo de credibilidad de una submuestra, de modo
  que se ve si los puntos que se salen de la diagonal están cubiertos.

Los puntos de train y de test van con marcador distinto: si la nube de test se
descuelga de la diagonal mientras la de train no, es deriva del bloque de
prueba y no error de ajuste.


In [ ]:
# [CONFIG] un origen de cada CADA_BARRA lleva barra de intervalo.
CADA_BARRA = 12

for M in M_OK:
    e = EST[M]
    n_comp = e["n_components"]
    fig, axes = plt.subplots(1, n_comp, figsize=(4.3 * n_comp, 4.3), squeeze=False)
    axes = axes[0]
    for k, ax in enumerate(axes):
        y, yh = e["Y_obs"][:, k], e["Y_hat"][:, k]
        lo, hi = e["li_s"][:, k], e["ls_s"][:, k]
        for m, nombre, color, marca in ((es_train, "train", "#2c7fb8", "o"),
                                        (~es_train, "test", "#c0392b", "^")):
            ax.scatter(y[m], yh[m], s=9, alpha=0.45, c=color, marker=marca,
                       linewidths=0, label=nombre)
        sel = np.arange(0, n_orig, CADA_BARRA)
        ax.vlines(y[sel], lo[sel], hi[sel], color="0.5", lw=0.7, alpha=0.6,
                  zorder=0)

        lim = [float(min(y.min(), lo.min())), float(max(y.max(), hi.max()))]
        ax.plot(lim, lim, "k--", lw=1.2, zorder=3, label="diagonal $y=x$")
        pend, inter = np.polyfit(y, yh, 1)
        ax.plot(lim, [pend * lim[0] + inter, pend * lim[1] + inter],
                color="#2ca25f", lw=1.3, zorder=3,
                label=f"ajuste (pendiente {pend:.2f})")
        ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal", "box")

        r2_te = 1.0 - (((y - yh) ** 2)[~es_train].sum()
                       / max(float(((y[~es_train] - y[~es_train].mean()) ** 2).sum()), 1e-12))
        cob = float(((y >= lo) & (y <= hi))[~es_train].mean())
        ax.set_title(f"FPC {e['component_idx'][k] + 1}\n"
                     f"$R^2$ test {r2_te:+.3f} · pendiente {pend:.2f} · "
                     f"PICP test {cob:.3f}", fontsize=9)
        ax.set_xlabel(rf"$\xi_{{{e['component_idx'][k] + 1}}}$ observado")
        if k == 0:
            ax.set_ylabel(r"$\hat\xi$ (media predictiva)")
            ax.legend(fontsize=7, loc="upper left")
    fig.suptitle(f"M={M} · $\\hat\\xi$ contra $\\xi$ · barras = IC "
                 f"{NIVEL:.0%} (una de cada {CADA_BARRA})", fontsize=12)
    fig.tight_layout()
    fig.savefig(e["paths"]["out_report"] / "94_scores_dispersion_vs_real.png",
                dpi=150, bbox_inches="tight")
    plt.show()


